# Comparaison de modèles — HPP sévère

Notebook **synthèse** (parcours certification). Les GridSearch complets restent dans `_archive/03|04|05_*.ipynb`.

**Contexte** : cible binaire `hpp_trans` (~2 % positifs) → priorité **recall**.

**Approche Jedha** :
1. Baseline sans rééquilibrage
2. Rééquilibrage (`class_weight`, SMOTE, RandomUnderSampler)
3. Comparaison LogReg / Random Forest / XGBoost
4. Tracking MLflow + export `joblib` pour l’API

## 1. Setup & preprocessor

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imblearn non installé — comparaison légère sans SMOTE/Under.")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost non installé — XGB ignoré dans la démo locale.")

DATA = Path("..") / "00_Data"
TARGET = "hpp_trans"

QUANT = ["age_m", "taille_mere", "bmi", "parite", "hosp_m_g", "dsm_g", "nbilan", "nsej18", "terme"]
BINARY = ["tabac", "hta_tot", "cholestase", "hellp", "creta", "ut_cica", "cortico", "pma", "bilan"]
NOMINAL = ["diabete", "AMP"]
ORDINAL = ["preecl", "g_type"]
FEATURES = QUANT + BINARY + NOMINAL + ORDINAL


def build_preprocessor(quant, binary, nominal, ordinal):
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    nominal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric, quant),
        ("bin", "passthrough", binary),
        ("nom", nominal_pipe, nominal),
        ("ord", ordinal_pipe, ordinal),
    ])

## 2. Résultats sur base complète (référence certification)

Métriques issues des expériences archivées (optimisées **recall**, rééquilibrage testé).
Tracking : https://thibautmodrin-mlflow.hf.space/

In [ ]:
results_full = pd.DataFrame([
    {"modèle": "LogReg + SMOTE", "recall": 0.69, "precision": 0.08, "commentaire": "Simple & interprétable"},
    {"modèle": "Random Forest", "recall": 0.65, "precision": 0.09, "commentaire": "Non-linéaire, robuste"},
    {"modèle": "XGBoost", "recall": 0.66, "precision": 0.09, "commentaire": "Proche RF"},
])
results_full

### Lecture métier

- Les 3 familles convergent (~65–69 % recall) : le **plafond** vient surtout du signal rare, pas du choix d’algo.
- La précision reste basse (~8–9 %) : beaucoup de faux positifs → le modèle est un **outil d’alerte**, pas un diagnostic.
- **Prod** : LogReg (interprétabilité clinique) exportée en `joblib` pour l’API FastAPI / Streamlit.

## 3. Démo pipeline (extrait portfolio)

L’extrait repo est trop petit pour rejouer SMOTE / GridSearch.  
Si la base complète est disponible localement, le bloc suivant lance une **comparaison légère** (sans GridSearch).

In [ ]:
# Option : pointer vers la base nettoyée hors repo
FULL_DATA = Path("Project/clean/Bourgogne20132023_clean.csv")  # chemin historique Jedha
EXTRACT = DATA / "extract_database.csv"

src = FULL_DATA if FULL_DATA.exists() else EXTRACT
df = pd.read_csv(src, low_memory=False)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

cols = [c for c in FEATURES + [TARGET] if c in df.columns]
df = df[cols].dropna()
print(f"Source : {src.name} → {df.shape[0]} lignes après dropna")

quant = [c for c in QUANT if c in df.columns]
binary = [c for c in BINARY if c in df.columns]
nominal = [c for c in NOMINAL if c in df.columns]
ordinal = [c for c in ORDINAL if c in df.columns]
feat_cols = quant + binary + nominal + ordinal

can_train = TARGET in df.columns and df[TARGET].nunique() >= 2 and len(df) >= 200
print("Entraînement local possible :", can_train)

In [ ]:
def evaluate(pipe, X_test, y_test):
    y_pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None
    row = {
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 3),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 3),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 3),
    }
    if proba is not None and y_test.nunique() > 1:
        row["roc_auc"] = round(roc_auc_score(y_test, proba), 3)
    return row


if can_train:
    X = df[feat_cols]
    y = df[TARGET].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    pre = build_preprocessor(quant, binary, nominal, ordinal)

    candidates = {
        "LogReg_baseline": Pipeline([
            ("preprocessor", pre),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]),
        "LogReg_balanced": Pipeline([
            ("preprocessor", pre),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]),
        "RandomForest": Pipeline([
            ("preprocessor", pre),
            ("classifier", RandomForestClassifier(
                n_estimators=200, class_weight="balanced_subsample", random_state=42, n_jobs=-1
            )),
        ]),
    }

    if HAS_IMBLEARN:
        candidates["LogReg_SMOTE"] = ImbPipeline([
            ("preprocessor", build_preprocessor(quant, binary, nominal, ordinal)),
            ("sampler", SMOTE(random_state=42)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ])
        candidates["LogReg_Under"] = ImbPipeline([
            ("preprocessor", build_preprocessor(quant, binary, nominal, ordinal)),
            ("sampler", RandomUnderSampler(random_state=42)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ])

    if HAS_XGB:
        # scale_pos_weight ≈ n_neg / n_pos
        spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        candidates["XGBoost"] = Pipeline([
            ("preprocessor", build_preprocessor(quant, binary, nominal, ordinal)),
            ("classifier", XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                scale_pos_weight=spw,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            )),
        ])

    rows = []
    for name, model in candidates.items():
        model.fit(X_train, y_train)
        metrics = evaluate(model, X_test, y_test)
        metrics["modèle"] = name
        rows.append(metrics)
        print(name, metrics)

    pd.DataFrame(rows).set_index("modèle")
else:
    print(
        "Pas assez de données dans le portfolio pour entraîner.\n"
        "→ Voir le tableau de référence §2 et les notebooks `_archive/`.\n"
        "→ Artefact prod déjà versionné : ../app/model/artifacts/model.joblib"
    )

## 4. Industrialisation

| Couche | Lien |
|--------|------|
| Artefact | `../app/model/artifacts/model.joblib` + `feature_order.pkl` |
| API | https://hpp-api.onrender.com/docs |
| Streamlit | https://thibautmodrin-hpp-prediction.hf.space |
| MLflow | https://thibautmodrin-mlflow.hf.space/ |

Pour ré-exporter un modèle après entraînement local :

```python
import joblib
from pathlib import Path

ART = Path("../app/model/artifacts")
joblib.dump(best_pipeline, ART / "model.joblib")
joblib.dump(feat_cols, ART / "feature_order.pkl")
```

## 5. Conclusion

1. **dropna** + features pré-accouchement uniquement.
2. Rééquilibrage indispensable (sinon recall ≈ 5 %).
3. LogReg / RF / XGB **équivalents** sur recall → choix prod = **LogReg** (lisibilité).
4. Déploiement = API + Streamlit + MLflow (hors notebooks).